# Genome PRD v1 — 02 Temporal splits

Apply and audit the canonical half-open train, validation, and test windows. No dates are duplicated here.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

here = Path.cwd().resolve()
ROOT = next(path for path in (here, *here.parents) if (path / 'src/training/prd_config.py').is_file())
sys.path.insert(0, str(ROOT))
from src.training import prd_config

prd_config.validate_split_contract()
events = pd.read_parquet(prd_config.FEATURE_ARTIFACT_PATH, columns=['ratingEventId', 'timestamp'])
ratings = pd.read_parquet(prd_config.RATINGS_SOURCE_PATH, columns=['rating'])['rating'].to_numpy(dtype=np.float32, copy=False)
event_ids = events['ratingEventId'].to_numpy(dtype=np.uint64, copy=False)
labels = (ratings[event_ids - 1] >= prd_config.PRD_TARGET_THRESHOLD).astype(np.int8)

## Split coverage, prevalence, and temporal ordering

In [2]:
masks = {}
rows = []
for name, (start, end) in prd_config.PRD_SPLITS.items():
    mask = np.ones(len(events), dtype=bool)
    if start is not None: mask &= events['timestamp'].to_numpy() >= start.to_datetime64()
    if end is not None: mask &= events['timestamp'].to_numpy() < end.to_datetime64()
    masks[name] = mask
    split_time = events.loc[mask, 'timestamp']
    rows.append({'split': name, 'start_inclusive': start, 'end_exclusive': end, 'actual_min': split_time.min(), 'actual_max': split_time.max(), 'rows': int(mask.sum()), 'prevalence': float(labels[mask].mean())})
summary = pd.DataFrame(rows).set_index('split')
display(summary)
assert sum(int(mask.sum()) for mask in masks.values()) == len(events)
assert not any(np.any(masks[left] & masks[right]) for left, right in [('train', 'validation'), ('train', 'test'), ('validation', 'test')])
assert summary.loc['train', 'actual_max'] < summary.loc['validation', 'actual_min'] < summary.loc['test', 'actual_min']
print('Coverage, non-overlap, and temporal ordering: PASS')

,start_inclusive,end_exclusive,actual_min,actual_max,rows,prevalence
split,,,,,,
train,NaT,2012-01-01,1995-01-09 11:46:44,2011-12-31 23:59:55,17822773,0.498194
validation,2012-01-01,2014-01-01,2012-01-01 00:00:40,2013-12-31 23:59:59,1330716,0.519539
test,2014-01-01,NaT,2014-01-01 00:00:04,2015-03-31 06:40:02,846774,0.501728


Coverage, non-overlap, and temporal ordering: PASS
